## Отчеты ФРСП

## ka_3_2

In [1]:
import pyodbc
from openpyxl import load_workbook

# =====================================================
# НАСТРОЙКИ
# =====================================================

DATE_FROM = "2026-07-01"
DATE_TO = "2026-10-1"  # не включительно

EXCEL_FILE = r"C:\Share\ORION\Отчет ФРСП-Orion.xlsx"
SHEET_NAME = "ka_3_2"

# =====================================================
# ПОДКЛЮЧЕНИЕ
# =====================================================

# Реквизиты CRM берём из scripts/config.py (CRM_DB), чтобы не хранить пароль в ноутбуке.
# config.py не импортируем (у него тяжёлые побочные эффекты) — читаем только литерал CRM_DB.
import ast, os

def load_crm_db():
    for path in ("config.py", os.path.join("scripts", "config.py")):
        if os.path.exists(path):
            tree = ast.parse(open(path, encoding="utf-8").read())
            for node in tree.body:
                if isinstance(node, ast.Assign) and any(
                    isinstance(t, ast.Name) and t.id == "CRM_DB" for t in node.targets
                ):
                    return ast.literal_eval(node.value)
    raise RuntimeError("CRM_DB не найден в scripts/config.py")

CRM_DB = load_crm_db()

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={CRM_DB['server']};"
    f"DATABASE={CRM_DB['database']};"
    f"UID={CRM_DB['username']};"
    f"PWD={CRM_DB['password']};"
)

cursor = conn.cursor()

wb = load_workbook(EXCEL_FILE)
ws = wb[SHEET_NAME]

# =====================================================
# ОБЩАЯ ФУНКЦИЯ ЗАПИСИ
# Только N-T
# =====================================================

MONEY_COLS = ["N", "O", "P", "Q", "R"]
COUNT_COLS = ["S", "T"]
ALL_COLS = MONEY_COLS + COUNT_COLS

def write_row(excel_row, r):
    for col in MONEY_COLS:
        ws[f"{col}{excel_row}"] = round((getattr(r, col) or 0) / 1000, 2)

    for col in COUNT_COLS:
        ws[f"{col}{excel_row}"] = getattr(r, col) or 0

def clear_row(excel_row):
    for col in ALL_COLS:
        ws[f"{col}{excel_row}"] = 0

def get_cell_value(cell):
    value = ws[cell].value
    return value if isinstance(value, (int, float)) else 0


# =====================================================
# БЛОК 1
# Строки 45-52
# Задолженность, приобретенная в отчетном периоде
# =====================================================

def get_acquired_debt_row(dpd_from=None, dpd_to=None):
    dpd_filter = ""
    params = [DATE_FROM, DATE_TO]

    if dpd_from is not None and dpd_to is not None:
        dpd_filter = "AND l.F63 BETWEEN ? AND ?"
        params.extend([dpd_from, dpd_to])

    sql = f"""
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F300,0)) AS N,
        SUM(ISNULL(l.F296,0)) AS O,
        SUM(ISNULL(l.F303,0)) AS P,
        0 AS Q,
        SUM(ISNULL(l.F299,0)) AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246

    WHERE
        l.F68 >= @DateFrom
        AND l.F68 < @DateTo
        AND l.F209 LIKE N'%Orion%'
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
        {dpd_filter};
    """

    cursor.execute(sql, *params)
    return cursor.fetchone()


row_45 = get_acquired_debt_row()
write_row(45, row_45)
write_row(46, row_45)

for excel_row, dpd_from, dpd_to in [
    (47, 0, 90),
    (48, 91, 180),
    (49, 181, 365),
    (50, 366, 730),
    (51, 731, 1825),
    (52, 1826, 999999),
]:
    write_row(excel_row, get_acquired_debt_row(dpd_from, dpd_to))

print("Строки 45-52 заполнены.")


# =====================================================
# БЛОК 2
# Строки 67-74
# Стоимость приобретенной задолженности / F322
# =====================================================

def get_purchase_price_row(dpd_from=None, dpd_to=None):
    dpd_filter = ""
    params = [DATE_FROM, DATE_TO]

    if dpd_from is not None and dpd_to is not None:
        dpd_filter = "AND l.F63 BETWEEN ? AND ?"
        params.extend([dpd_from, dpd_to])

    debt_total = "NULLIF(ISNULL(l.F300,0)+ISNULL(l.F296,0)+ISNULL(l.F303,0)+ISNULL(l.F299,0),0)"

    sql = f"""
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F322,0) * ISNULL(l.F300,0) / {debt_total}) AS N,
        SUM(ISNULL(l.F322,0) * ISNULL(l.F296,0) / {debt_total}) AS O,
        SUM(ISNULL(l.F322,0) * ISNULL(l.F303,0) / {debt_total}) AS P,
        0 AS Q,
        SUM(ISNULL(l.F322,0) * ISNULL(l.F299,0) / {debt_total}) AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246

    WHERE
        l.F68 >= @DateFrom
        AND l.F68 < @DateTo
        AND l.F209 LIKE N'%Orion%'
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
        {dpd_filter};
    """

    cursor.execute(sql, *params)
    return cursor.fetchone()


row_67 = get_purchase_price_row()
write_row(67, row_67)
write_row(68, row_67)

for excel_row, dpd_from, dpd_to in [
    (69, 0, 90),
    (70, 91, 180),
    (71, 181, 365),
    (72, 366, 730),
    (73, 731, 1825),
    (74, 1826, 999999),
]:
    write_row(excel_row, get_purchase_price_row(dpd_from, dpd_to))

print("Строки 67-74 заполнены.")


# =====================================================
# БЛОК 3
# Строки 114-123
# Банкрот / списанная задолженность
# =====================================================

def get_bankruptcy_row(overdue_from=None, overdue_to=None):
    overdue_filter = ""
    params = [DATE_FROM, DATE_TO]

    if overdue_from is not None and overdue_to is not None:
        overdue_filter = "AND ISNULL(l.Overdue,0) BETWEEN ? AND ?"
        params.extend([overdue_from, overdue_to])

    sql = f"""
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F32,0)) AS N,
        SUM(ISNULL(l.F8,0)) AS O,
        SUM(ISNULL(l.F38,0)) AS P,
        0 AS Q,
        0 AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    JOIN states s WITH (NOLOCK) ON s.ID = l.State
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246

    WHERE
        l.F209 LIKE N'%Orion%'
        AND s.Caption = N'Банкрот'
        AND c.F336 >= @DateFrom
        AND c.F336 < @DateTo
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
        {overdue_filter};
    """

    cursor.execute(sql, *params)
    return cursor.fetchone()


row_114 = get_bankruptcy_row()

write_row(114, row_114)
write_row(115, row_114)
write_row(116, row_114)

for excel_row, overdue_from, overdue_to in [
    (117, 0, 0),
    (118, 1, 90),
    (119, 91, 180),
    (120, 181, 365),
    (121, 366, 730),
    (122, 731, 1825),
    (123, 1826, 999999),
]:
    write_row(excel_row, get_bankruptcy_row(overdue_from, overdue_to))

print("Строки 114-123 заполнены.")


# =====================================================
# ДОП. БЛОК
# Строки 156-157
# Сделки по дате l.F374 в отчетном периоде
# 157 = дубль 156
# Строка 20 дополнительно минусует строку 156
# =====================================================

def get_f374_period_row():
    sql = """
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F32,0)) AS N,
        SUM(ISNULL(l.F8,0)) AS O,
        SUM(ISNULL(l.F38,0)) AS P,
        0 AS Q,
        0 AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246

    WHERE
        l.F209 LIKE N'%Orion%'
        AND l.F374 >= @DateFrom
        AND l.F374 < @DateTo
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%';
    """

    cursor.execute(sql, DATE_FROM, DATE_TO)
    return cursor.fetchone()


row_156 = get_f374_period_row()

write_row(156, row_156)
write_row(157, row_156)

print("Строки 156-157 заполнены.")

# =====================================================
# БЛОК 4
# Строки 165-174
# Остаток задолженности
# =====================================================

def get_current_balance_row(overdue_from=None, overdue_to=None):
    loan_filter = ""
    iin_filter = ""

    if overdue_from is not None and overdue_to is not None:
        loan_filter = f"AND ISNULL(l.Overdue,0) BETWEEN {overdue_from} AND {overdue_to}"
        iin_filter = f"AND ib.MaxOverdue BETWEEN {overdue_from} AND {overdue_to}"

    sql = f"""
    WITH IinOwner AS (
        SELECT
            IIN,
            MaxOverdue
        FROM (
            SELECT
                c.F293 AS IIN,
                MAX(ISNULL(l.Overdue,0)) OVER (PARTITION BY c.F293) AS MaxOverdue,
                ROW_NUMBER() OVER (
                    PARTITION BY c.F293
                    ORDER BY SUM(ISNULL(l.F32,0)+ISNULL(l.F8,0)+ISNULL(l.F38,0)) DESC
                ) AS rn
            FROM loans l WITH (NOLOCK)
            JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
            JOIN states s WITH (NOLOCK) ON s.ID = l.State
            LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246
            WHERE
                l.F209 LIKE N'%Orion%'
                AND s.Caption NOT IN (N'Обратный выкуп', N'Погашен', N'Банкрот')
                AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
                AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
            GROUP BY
                c.F293,
                l.Overdue
        ) x
        WHERE rn = 1
    )

    SELECT
        SUM(CASE WHEN 1=1 {loan_filter} THEN ISNULL(l.F32,0) ELSE 0 END) AS N,
        SUM(CASE WHEN 1=1 {loan_filter} THEN ISNULL(l.F8,0) ELSE 0 END) AS O,
        SUM(CASE WHEN 1=1 {loan_filter} THEN ISNULL(l.F38,0) ELSE 0 END) AS P,
        0 AS Q,
        0 AS R,
        COUNT(DISTINCT CASE WHEN 1=1 {loan_filter} THEN l.EID END) AS S,

        (
            SELECT COUNT(DISTINCT ib.IIN)
            FROM IinOwner ib
            WHERE 1=1
            {iin_filter}
        ) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    JOIN states s WITH (NOLOCK) ON s.ID = l.State
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246

    WHERE
        l.F209 LIKE N'%Orion%'
        AND s.Caption NOT IN (N'Обратный выкуп', N'Погашен', N'Банкрот')
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%';
    """

    cursor.execute(sql)
    return cursor.fetchone()


row_165 = get_current_balance_row()

write_row(165, row_165)
write_row(166, row_165)
write_row(167, row_165)

clear_row(168)

for excel_row, overdue_from, overdue_to in [
    (169, 0, 90),
    (170, 91, 180),
    (171, 181, 365),
    (172, 366, 730),
    (173, 731, 1825),
    (174, 1826, 999999),
]:
    write_row(excel_row, get_current_balance_row(overdue_from, overdue_to))

print("Строки 165-174 заполнены.")


# =====================================================
# БЛОК 5
# Строки 89-97
# Погашенная задолженность
# 89 = 20 + 45 - 114 - 165
# 90 = дубль 89
# 91 = 0
# 92-97 = пропорционально строкам 169-174 к строке 165
# =====================================================

for col in ALL_COLS:
    ws[f"{col}89"] = (
        get_cell_value(f"{col}20")
        + get_cell_value(f"{col}45")
        - get_cell_value(f"{col}114")
        - get_cell_value(f"{col}156")
        - get_cell_value(f"{col}165")
    )

for col in ALL_COLS:
    ws[f"{col}90"] = get_cell_value(f"{col}89")

for col in ALL_COLS:
    ws[f"{col}91"] = 0

bucket_map = [
    (92, 169),
    (93, 170),
    (94, 171),
    (95, 172),
    (96, 173),
    (97, 174),
]

for target_row, balance_bucket_row in bucket_map:
    for col in MONEY_COLS:
        total_paid = get_cell_value(f"{col}89")
        total_balance = get_cell_value(f"{col}165")
        bucket_balance = get_cell_value(f"{col}{balance_bucket_row}")

        if total_balance != 0:
            ws[f"{col}{target_row}"] = round(total_paid * bucket_balance / total_balance, 2)
        else:
            ws[f"{col}{target_row}"] = 0

for col in COUNT_COLS:
    total_paid = int(round(get_cell_value(f"{col}89")))
    total_balance = get_cell_value(f"{col}165")

    raw_values = []

    for target_row, balance_bucket_row in bucket_map:
        bucket_balance = get_cell_value(f"{col}{balance_bucket_row}")
        raw = total_paid * bucket_balance / total_balance if total_balance != 0 else 0
        raw_values.append((target_row, raw))

    floors = [(row, int(raw)) for row, raw in raw_values]
    current_sum = sum(v for _, v in floors)
    diff = total_paid - current_sum

    remainders = sorted(
        [(row, raw - int(raw)) for row, raw in raw_values],
        key=lambda x: x[1],
        reverse=True
    )

    result = dict(floors)

    for i in range(diff):
        result[remainders[i][0]] += 1

    for target_row, _ in bucket_map:
        ws[f"{col}{target_row}"] = result[target_row]

print("Строки 89-97 заполнены.")

# =====================================================
# Строки 207-209
# Orion
#
# 207 = 208 + 209
# 208 = прощенная / списанная задолженность
# 209 = частично или полностью погашенная задолженность
#
# Исключаем продукт Vivus
#
# payment:
# F100 = ОД
# F106 = Вознаграждение
# F102 = Пеня
# F104 = Госпошлина, в отчет не берем
# =====================================================

def get_forgiven_debt_row():
    sql = """
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F32,0)) AS N,
        SUM(ISNULL(l.F8,0)) AS O,
        SUM(ISNULL(l.F38,0)) AS P,
        0 AS Q,
        0 AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    JOIN states s WITH (NOLOCK) ON s.ID = l.State
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246
    WHERE
        l.F209 LIKE N'%Orion%'
        AND l.F68 >= @DateFrom
        AND l.F68 < @DateTo
        AND s.Caption = N'Банкрот'
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%';
    """

    cursor.execute(sql, DATE_FROM, DATE_TO)
    return cursor.fetchone()


def get_paid_debt_row():
    sql = """
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(p.F100,0)) AS N,
        SUM(ISNULL(p.F106,0)) AS O,
        SUM(ISNULL(p.F102,0)) AS P,
        0 AS Q,
        0 AS R,
        COUNT(DISTINCT l.EID) AS S,
        COUNT(DISTINCT c.F293) AS T

    FROM payment p WITH (NOLOCK)
    JOIN loans l WITH (NOLOCK) ON l.ID = p.LID
    JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
    LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246
    WHERE
        l.F209 LIKE N'%Orion%'
        AND l.F68 >= @DateFrom
        AND l.F68 < @DateTo
        AND p.Created >= @DateFrom
        AND p.Created < @DateTo
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%';
    """

    cursor.execute(sql, DATE_FROM, DATE_TO)
    return cursor.fetchone()


# 208 = прощенная / списанная задолженность
write_row(208, get_forgiven_debt_row())

# 209 = частично или полностью погашенная задолженность
write_row(209, get_paid_debt_row())

# 207 = сумма 208 + 209
for col in ALL_COLS:
    ws[f"{col}207"] = get_cell_value(f"{col}208") + get_cell_value(f"{col}209")

print("Строки 207-209 заполнены.")


Строки 45-52 заполнены.


Строки 67-74 заполнены.


Строки 114-123 заполнены.
Строки 156-157 заполнены.


Строки 165-174 заполнены.
Строки 89-97 заполнены.
Строки 207-209 заполнены.


## ka_3_4

In [2]:
# =====================================================
# ЛИСТ ka_3_4
# Блоки:
# - Досудебное взыскание: AT:AZ
# - Решение суда / нотариальная надпись: BA:BG
# - Исполнительное производство: BH:BN
#
# Orion:
# - заполняется строка 21
# - продукт содержит MM / ММ
# - строка 20 = дубль строки 21
# =====================================================

ws2 = wb["ka_3_4"]

def write_ka34_row(excel_row, r):
    # Досудебное взыскание
    ws2[f"AT{excel_row}"] = round((r.AT or 0) / 1000, 2)
    ws2[f"AU{excel_row}"] = round((r.AU or 0) / 1000, 2)
    ws2[f"AV{excel_row}"] = round((r.AV or 0) / 1000, 2)
    ws2[f"AY{excel_row}"] = r.AY or 0
    ws2[f"AZ{excel_row}"] = r.AZ or 0

    # Решение суда / нотариальная надпись
    ws2[f"BA{excel_row}"] = round((r.BA or 0) / 1000, 2)
    ws2[f"BB{excel_row}"] = round((r.BB or 0) / 1000, 2)
    ws2[f"BC{excel_row}"] = round((r.BC or 0) / 1000, 2)
    ws2[f"BF{excel_row}"] = r.BF or 0
    ws2[f"BG{excel_row}"] = r.BG or 0

    # Исполнительное производство
    ws2[f"BH{excel_row}"] = round((r.BH or 0) / 1000, 2)
    ws2[f"BI{excel_row}"] = round((r.BI or 0) / 1000, 2)
    ws2[f"BJ{excel_row}"] = round((r.BJ or 0) / 1000, 2)
    ws2[f"BM{excel_row}"] = r.BM or 0
    ws2[f"BN{excel_row}"] = r.BN or 0


def get_ka34_mm_row():
    sql = """
    WITH Base AS (
        SELECT
            l.EID,
            c.F293 AS IIN,
            ISNULL(dF246.F249, N'') AS ProductName,
            s.Caption AS StateName,
            ISNULL(constF236.Caption, N'') AS AisStatus,
            ISNULL(l.F32,0) AS F32,
            ISNULL(l.F8,0) AS F8,
            ISNULL(l.F38,0) AS F38,
            ISNULL(l.F32,0) + ISNULL(l.F8,0) + ISNULL(l.F38,0) AS TotalDebt
        FROM loans l WITH (NOLOCK)
        JOIN clients c WITH (NOLOCK) ON c.ID = l.CID
        JOIN states s WITH (NOLOCK) ON s.ID = l.State
        LEFT JOIN Dictionary dF246 WITH (NOLOCK) ON dF246.ID = l.F246
        LEFT JOIN Constants constF236 WITH (NOLOCK) ON constF236.ID = l.F236
        WHERE
            l.F209 LIKE N'%Orion%'
            AND s.Caption NOT IN (N'Обратный выкуп', N'Погашен', N'Банкрот')
            AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
            AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
    ),

    Marked AS (
        SELECT *,
            CASE 
                WHEN StateName <> N'В графике'
                     AND AisStatus LIKE N'%На исполнении%' THEN 'EXEC'
                WHEN StateName <> N'В графике' THEN 'COURT'
                WHEN StateName = N'В графике' THEN 'PRE'
                ELSE NULL
            END AS BlockType,

            CASE 
                WHEN StateName <> N'В графике'
                     AND AisStatus LIKE N'%На исполнении%' THEN 1
                WHEN StateName <> N'В графике' THEN 2
                WHEN StateName = N'В графике' THEN 3
                ELSE 99
            END AS BlockPriority
        FROM Base
    ),

    IinOwner AS (
        SELECT
            IIN,
            BlockType,
            ProductName
        FROM (
            SELECT
                IIN,
                BlockType,
                ProductName,
                ROW_NUMBER() OVER (
                    PARTITION BY IIN
                    ORDER BY 
                        MIN(BlockPriority),
                        SUM(TotalDebt) DESC
                ) AS rn
            FROM Marked
            WHERE BlockType IS NOT NULL
            GROUP BY IIN, BlockType, ProductName
        ) x
        WHERE rn = 1
    )

    SELECT
        -- Досудебное взыскание
        SUM(CASE WHEN m.BlockType = 'PRE' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F32 ELSE 0 END) AS AT,

        SUM(CASE WHEN m.BlockType = 'PRE' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F8 ELSE 0 END) AS AU,

        SUM(CASE WHEN m.BlockType = 'PRE' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F38 ELSE 0 END) AS AV,

        COUNT(DISTINCT CASE WHEN m.BlockType = 'PRE' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.EID END) AS AY,

        COUNT(DISTINCT CASE WHEN o.BlockType = 'PRE' 
                  AND (o.ProductName LIKE N'%MM%' OR o.ProductName LIKE N'%ММ%')
                 THEN o.IIN END) AS AZ,

        -- Суд / нотариальная надпись
        SUM(CASE WHEN m.BlockType = 'COURT' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F32 ELSE 0 END) AS BA,

        SUM(CASE WHEN m.BlockType = 'COURT' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F8 ELSE 0 END) AS BB,

        SUM(CASE WHEN m.BlockType = 'COURT' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F38 ELSE 0 END) AS BC,

        COUNT(DISTINCT CASE WHEN m.BlockType = 'COURT' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.EID END) AS BF,

        COUNT(DISTINCT CASE WHEN o.BlockType = 'COURT' 
                  AND (o.ProductName LIKE N'%MM%' OR o.ProductName LIKE N'%ММ%')
                 THEN o.IIN END) AS BG,

        -- Исполнительное производство
        SUM(CASE WHEN m.BlockType = 'EXEC' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F32 ELSE 0 END) AS BH,

        SUM(CASE WHEN m.BlockType = 'EXEC' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F8 ELSE 0 END) AS BI,

        SUM(CASE WHEN m.BlockType = 'EXEC' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.F38 ELSE 0 END) AS BJ,

        COUNT(DISTINCT CASE WHEN m.BlockType = 'EXEC' 
                  AND (m.ProductName LIKE N'%MM%' OR m.ProductName LIKE N'%ММ%')
                 THEN m.EID END) AS BM,

        COUNT(DISTINCT CASE WHEN o.BlockType = 'EXEC' 
                  AND (o.ProductName LIKE N'%MM%' OR o.ProductName LIKE N'%ММ%')
                 THEN o.IIN END) AS BN

    FROM Marked m
    LEFT JOIN IinOwner o
        ON o.IIN = m.IIN;
    """

    cursor.execute(sql)
    return cursor.fetchone()


# строка 21 = Orion / MM / ММ
row_21 = get_ka34_mm_row()
write_ka34_row(21, row_21)

# строка 20 = дубль строки 21
write_ka34_row(20, row_21)

print("Лист ka_3_4 заполнен.")

Лист ka_3_4 заполнен.


## ka_3_5

In [3]:
# =====================================================
# ЛИСТ ka_3_5
# Строки 19-28 и 61-70
# Реструктуризация
# Условия:
# l.F220 = 1
# l.F323 IS NOT NULL
# F258 ранее периода / в отчетном периоде
# Бакеты по текущему Overdue
#
# Orion:
# - исключаем продукт Vivus
# - заполняем только колонки H-L
# =====================================================

ws3 = wb["ka_3_5"]

def write_ka35_row(excel_row, r):
    ws3[f"H{excel_row}"] = round((r.H or 0) / 1000, 2)  # F34
    ws3[f"I{excel_row}"] = round((r.I or 0) / 1000, 2)  # F32
    ws3[f"J{excel_row}"] = round((r.J or 0) / 1000, 2)  # F8
    ws3[f"K{excel_row}"] = r.K or 0                    # EID
    ws3[f"L{excel_row}"] = r.L or 0                    # IIN


def clear_ka35_row(excel_row):
    for col in ["H", "I", "J", "K", "L"]:
        ws3[f"{col}{excel_row}"] = 0


def get_ka35_restruct_row(date_mode, overdue_from=None, overdue_to=None):
    overdue_filter = ""
    params = [DATE_FROM, DATE_TO]

    if date_mode == "before":
        date_filter = "AND l.F258 < @DateFrom"
    elif date_mode == "period":
        date_filter = "AND l.F258 >= @DateFrom AND l.F258 < @DateTo"
    else:
        raise ValueError("date_mode должен быть before или period")

    if overdue_from is not None and overdue_to is not None:
        overdue_filter = "AND ISNULL(l.Overdue,0) BETWEEN ? AND ?"
        params.extend([overdue_from, overdue_to])

    sql = f"""
    DECLARE @DateFrom date = ?;
    DECLARE @DateTo date = ?;

    SELECT
        SUM(ISNULL(l.F34,0)) AS H,
        SUM(ISNULL(l.F32,0)) AS I,
        SUM(ISNULL(l.F8,0)) AS J,
        COUNT(DISTINCT l.EID) AS K,
        COUNT(DISTINCT c.F293) AS L

    FROM loans l WITH (NOLOCK)
    JOIN clients c WITH (NOLOCK)
        ON c.ID = l.CID

    JOIN states s WITH (NOLOCK)
        ON s.ID = l.State

    LEFT JOIN Dictionary dF246 WITH (NOLOCK)
        ON dF246.ID = l.F246

    WHERE
        l.F209 LIKE N'%Orion%'
        AND s.Caption NOT IN (
            N'Обратный выкуп',
            N'Погашен',
            N'Банкрот'
        )
        AND l.F220 = 1
        AND l.F323 IS NOT NULL
        AND ISNULL(dF246.F249,N'') NOT LIKE N'%Vivus%'
        AND ISNULL(dF246.F250,N'') NOT LIKE N'%Vivus%'
        {date_filter}
        {overdue_filter};
    """

    cursor.execute(sql, *params)
    return cursor.fetchone()


# =====================================================
# Первый блок: строки 19-28
# F258 ранее отчетного периода
# =====================================================

row_19 = get_ka35_restruct_row("before")
write_ka35_row(19, row_19)
write_ka35_row(20, row_19)
write_ka35_row(21, row_19)

clear_ka35_row(22)

for excel_row, overdue_from, overdue_to in [
    (23, 0, 90),
    (24, 91, 180),
    (25, 181, 365),
    (26, 366, 730),
    (27, 731, 1825),
    (28, 1826, 999999),
]:
    write_ka35_row(excel_row, get_ka35_restruct_row("before", overdue_from, overdue_to))


# =====================================================
# Второй блок: строки 61-70
# F258 в отчетном периоде
# =====================================================

row_61 = get_ka35_restruct_row("period")
write_ka35_row(61, row_61)
write_ka35_row(62, row_61)
write_ka35_row(63, row_61)

clear_ka35_row(64)

for excel_row, overdue_from, overdue_to in [
    (65, 0, 90),
    (66, 91, 180),
    (67, 181, 365),
    (68, 366, 730),
    (69, 731, 1825),
    (70, 1826, 999999),
]:
    write_ka35_row(excel_row, get_ka35_restruct_row("period", overdue_from, overdue_to))

print("Лист ka_3_5 заполнен.")

Лист ka_3_5 заполнен.


## ka_7_uzfl

## СОХРАНЕНИЕ И ЗАКРЫТИЕ

In [4]:
# =====================================================
# СОХРАНЕНИЕ И ЗАКРЫТИЕ
# =====================================================

wb.save(EXCEL_FILE)

cursor.close()
conn.close()

print("Отчет ФРСП успешно сохранен.")

Отчет ФРСП успешно сохранен.
